In [ ]:
!pip install transformers datasets accelerate scikit-learn

In [ ]:
import numpy as np, pandas as pd, torch
from dataclasses import dataclass
from typing import Any
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, classification_report)
from transformers import (AutoFeatureExtractor, AutoModelForAudioClassification,
                          TrainingArguments, Trainer)

In [ ]:
CHECKPOINT = "microsoft/unispeech-sat-base"
MAX_LEN = 6 * 16000
PER_GROUP = 100
SEED = 42

### 2. Φόρτωση του Crema_D

In [ ]:
dataset = load_dataset("AbstractTTS/CREMA-D", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
print(dataset[0]["file"], dataset[0]["major_emotion"])

In [ ]:
### 3. Ετικέτες και ομιλητές
labels = sorted(set(dataset["major_emotion"]))   # anger, disgust, fear, happy, neutral, sad
num_labels = len(labels)
label2id = {n: i for i, n in enumerate(labels)}
id2label = {i: n for n, i in label2id.items()}
print(label2id)

def add_speaker(batch):
    batch["actor_id"] = batch["file"].split("_")[0]
    return batch

dataset = dataset.map(add_speaker, load_from_cache_file=False)
print(len(set(dataset["actor_id"])), "μοναδικοί ομιλητές")

### 3. Split ανεξάρτητο ως προς τον ομιλητή

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["actor_id"]))
dataset = DatasetDict({"train": dataset.select(train_idx), "test": dataset.select(test_idx)})

train_spk, test_spk = set(dataset["train"]["actor_id"]), set(dataset["test"]["actor_id"])
assert not (train_spk & test_spk), "Επικάλυψη ομιλητών!"
print(len(dataset["train"]), "train |", len(dataset["test"]), "test | test ομιλητές:", len(test_spk))

### 5. Προεπεξεργασία (feature extractor)

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(CHECKPOINT)
print("return_attention_mask:", feature_extractor.return_attention_mask)

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"],
        max_length=MAX_LEN, truncation=True
    ).input_values[0]
    batch["labels"] = label2id[batch["major_emotion"]]
    return batch

encoded_dataset = dataset.map(prepare_dataset,
                              remove_columns=dataset["train"].column_names,
                              load_from_cache_file=False)

### 6. Μοντέλο

In [ ]:
model = AutoModelForAudioClassification.from_pretrained(
    CHECKPOINT, num_labels=num_labels, label2id=label2id, id2label=id2label,
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()
print(type(model).__name__, [n for n, _ in model.named_children()])

### 7. Collator και μετρικές

In [ ]:
class DataCollatorSER:
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, features):
        batch = self.feature_extractor.pad(
            [{"input_values": f["input_values"]} for f in features],
            padding=True, return_tensors="pt",
        )
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,

    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,

    average_precision_score
)

from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)

    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")

    # PR-AUC (macro-averaged average precision)
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    accuracy = accuracy_score(labels_true, predictions)
    balanced_accuracy = balanced_accuracy_score(labels_true, predictions)
    precision = precision_score(labels_true, predictions, average="weighted", zero_division=0)
    recall = recall_score(labels_true, predictions, average="weighted", zero_division=0)
    f1 = f1_score(labels_true, predictions, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(labels_true, predictions)
    k_score = cohen_kappa_score(labels_true, predictions)
    logloss = log_loss(labels_true, probs, labels=list(range(num_labels)))

    cm = confusion_matrix(labels_true, predictions, labels=list(range(num_labels)))
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc" : mcc,
        "k_score" : k_score,
        "log_loss": logloss,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    }

### 8. Εκπαίδευση

In [ ]:
import math

BATCH_SIZE = 8
GRAD_ACCUM = 4
EPOCHS = 8

steps_per_epoch = math.ceil(len(encoded_dataset["train"]) / (BATCH_SIZE * GRAD_ACCUM))
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(0.1 * total_steps)          # 10% των συνολικών βημάτων
print(f"steps/epoch={steps_per_epoch} | total={total_steps} | warmup={warmup_steps}")

training_args = TrainingArguments(
    output_dir="./unispeech-sat-finetuned-ESD",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=EPOCHS,
    learning_rate=3e-5,
    warmup_steps=warmup_steps,
    fp16=False,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    save_only_model=True,
    seed=SEED,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    processing_class=feature_extractor,
    data_collator=DataCollatorSER(feature_extractor),
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
### 9. Τελική αξιολόγηση του καλύτερου μοντέλου
pred = trainer.predict(encoded_dataset["test"])
y_true, y_pred = pred.label_ids, np.argmax(pred.predictions, axis=-1)
print(classification_report(y_true, y_pred, target_names=labels, digits=4))
print(confusion_matrix(y_true, y_pred))

In [ ]:
### 9β. Πλήρες σύνολο μετρικών
probs = softmax(pred.predictions, axis=-1)
y_bin = label_binarize(y_true, classes=list(range(num_labels)))

full_metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "mcc": matthews_corrcoef(y_true, y_pred),
    "kappa": cohen_kappa_score(y_true, y_pred),
    "log_loss": log_loss(y_true, probs, labels=list(range(num_labels))),
    "roc_auc_macro_ovr": roc_auc_score(y_bin, probs, average="macro", multi_class="ovr"),
    "pr_auc_macro": average_precision_score(y_bin, probs, average="macro"),
}
for k, v in full_metrics.items():
    print(f"{k:22s} {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιουργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δημιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

### 10. Αποθήκευση στο Hugging Face Hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/Unispeech_SAT_fnetuned_esd_english"

model.push_to_hub(repo_id)
feature_extractor.push_to_hub(repo_id)
